# SPY Next-Day High-Volatility Risk Pipeline
**Stakeholder:** Portfolio risk manager. **Decision:** whether a next-day risk score warrants additional human review. This notebook runs the full lifecycle from committed raw data through model, uncertainty, report, and API evidence.

In [1]:
# --- run me first ---
from pathlib import Path
import os, sys
if Path.cwd().name == 'notebooks':
    os.chdir('..')
ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print('working from:', ROOT.name)

working from: project


## Stages 4–13: acquire, clean, feature, model, evaluate, and report
The default uses the committed raw snapshot for reproducibility. Pass `refresh=True` only when a live provider update is intended.

In [2]:
from src.pipeline import run_full_pipeline
result = run_full_pipeline(refresh=False)
result

{'raw_rows': 2512,
 'featured_rows': 2451,
 'train_rows': 1960,
 'test_rows': 491,
 'data_start': '2016-11-22',
 'data_end': '2026-08-25',
 'metrics': {'accuracy': 0.7331975560081466,
  'precision': 0.46875,
  'recall': 0.4878048780487805,
  'f1': 0.47808764940239046,
  'roc_auc': 0.7159994697773064,
  'majority_accuracy': 0.7494908350305499,
  'positive_rate': 0.2505091649694501,
  'confusion_matrix': [[300, 68], [63, 60]]},
 'bootstrap_roc_auc': {'metric': 'roc_auc',
  'estimate': 0.7159994697773064,
  'mean': 0.7171801254968674,
  'lo': 0.6605508323493654,
  'hi': 0.7680740049446013,
  'n_boot': 600}}

## Evaluation evidence
ROC-AUC measures probability ranking; F1 balances precision and recall at the 50% review threshold. The bootstrap interval, threshold scenarios, and trend subgroups prevent the point estimate from being presented without risk context.

In [3]:
import json
import pandas as pd
metrics = json.loads(Path('reports/model_metrics.json').read_text())
bootstrap = json.loads(Path('reports/bootstrap_roc_auc.json').read_text())
scenarios = pd.read_csv('reports/scenario_results.csv')
subgroups = pd.read_csv('reports/subgroup_results.csv')
print('Metrics:', metrics)
print('Bootstrap ROC-AUC:', bootstrap)
display(scenarios)
display(subgroups)

Metrics: {'accuracy': 0.7331975560081466, 'confusion_matrix': [[300, 68], [63, 60]], 'f1': 0.47808764940239046, 'majority_accuracy': 0.7494908350305499, 'positive_rate': 0.2505091649694501, 'precision': 0.46875, 'recall': 0.4878048780487805, 'roc_auc': 0.7159994697773064}
Bootstrap ROC-AUC: {'estimate': 0.7159994697773064, 'hi': 0.7680740049446013, 'lo': 0.6605508323493654, 'mean': 0.7171801254968674, 'metric': 'roc_auc', 'n_boot': 600}


,scenario,target_quantile,accuracy,precision,recall,f1,roc_auc,positive_rate
0,Trailing quantile 70%,0.70,0.708758,0.507692,0.455172,0.480000,0.705900,0.295316
1,Trailing quantile 75%,0.75,0.733198,0.468750,0.487805,0.478088,0.715999,0.250509
2,Trailing quantile 80%,0.80,0.790224,0.476190,0.510204,0.492611,0.749727,0.199593


,market_regime,rows,positive_rate,f1,roc_auc
0,Above 50-day trend,381,0.183727,0.213592,0.648186
1,Below 50-day trend,110,0.481818,0.662162,0.620655


## Stage 13: API test evidence
The saved model is loaded once when `app.py` starts. These calls prove health, metadata, a valid prediction, and JSON error handling for an invalid feature vector.

In [4]:
import subprocess, time, requests
from src.features import FEATURE_COLUMNS
server = subprocess.Popen([sys.executable, 'app.py'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(2)
if server.poll() is not None:
    raise RuntimeError('Flask API failed to start on port 5001')
try:
    base = 'http://127.0.0.1:5001'
    latest = pd.read_csv('data/processed/spy_features.csv').iloc[-1]
    payload = {'features': {name: float(latest[name]) for name in FEATURE_COLUMNS}}
    for label, response in [
        ('health', requests.get(base + '/health', timeout=5)),
        ('metadata', requests.get(base + '/metadata', timeout=5)),
        ('valid prediction', requests.post(base + '/predict', json=payload, timeout=5)),
        ('invalid prediction', requests.post(base + '/predict', json={'features': [1, 2]}, timeout=5)),
    ]:
        print(label, response.status_code, response.json())
finally:
    server.terminate(); server.wait(timeout=5)
    print('API stopped after verification')

health 200 {'status': 'ok', 'trained_through': '2024-09-09'}
metadata 200 {'features': ['return_1d', 'return_lag_1', 'momentum_5', 'momentum_20', 'volatility_5', 'volatility_20', 'volatility_60', 'range_pct', 'volume_change_5', 'price_vs_sma20', 'drawdown_60'], 'target': 'Next-day absolute return exceeds trailing 252-session 75th percentile', 'threshold': 0.5}
valid prediction 200 {'decision_threshold': 0.5, 'high_volatility_alert': False, 'high_volatility_probability': 0.4502794163715594}
invalid prediction 400 {'error': 'features must be a mapping or a list of 11 values'}
API stopped after verification


## Risk-aware conclusion
The model has useful ranking power (ROC-AUC about 0.716) but modest threshold classification performance and material subgroup variation. Use the probability to trigger review, not an automatic trade. The decision holds only while data remain fresh, the target definition matches policy, and rolling performance stays above monitoring thresholds. See `reports/final_report.md`, `docs/monitoring_plan.md`, and `docs/project_summary.md`.